In [36]:
import numpy as np
import pandas as pd
from time import sleep
from pathlib import Path
from bokeh.plotting import figure, show
from scipy.interpolate import make_interp_spline
from scipy.signal import savgol_filter
from bokeh.io import output_notebook, export_png
output_notebook()

Loading BokehJS ...

In [37]:
def normalize(df, norm_data):
	norm = 0
	for l in range(len(norm_data)):
		norm += norm_data[l]

	norm /= len(norm_data)
	df["y"] /= norm
	
	return df

def doThePlot(paths, laser_power, points):
	norm_curves = []
	minis = []
	maxis = []
	z = 1
	for i in range(len(paths)):
		rawdf = pd.read_csv(paths[z - 1])
		df = rawdf.drop(columns="y2")
		df = df.rename(columns={"y1": "y"})

		norm_data = df.iloc[-100:, 1].to_numpy()
		df = normalize(df, norm_data)

		norm_curves.append(df["y"])

		source = df.to_dict(orient="list")

		transmittance_diff_p_v = max(df["y"]) - min(df["y"])
		distance_diff_p_v = (df["x"][np.argmax(df["y"])] - df["x"][np.argmin(df["y"])]) * 10**-2

		n2 = (transmittance_diff_p_v * ((laser_wavelength * 10**-9)**2) * distance_diff_p_v) / (0.406 * 4 * np.pi * (laser_power * 10**-3) * (sample_length * 10**-3) * 1.7)

		minis.append(min(df["y"]))
		maxis.append(max(df["y"]))

		if z < 6:
			print(f"n2 {z}: {n2}")

		z += 1

	df = pd.DataFrame()

	for yc in norm_curves:
		df = pd.concat([df, yc], axis=1)

	df = df.mean(axis=1).to_frame(name = "y")

	rawdf = pd.read_csv(paths[0])
	df.insert(0, "x", rawdf["x"])

	if points != 0:
		df["y"] = savgol_filter(df["y"], window_length=points, polyorder=3)

	source = df.to_dict(orient="list")

	print(f"\nn2: {n2} m²/W")
	print(f"ΔTpv: {transmittance_diff_p_v}")
	print(f"Δzpv: {distance_diff_p_v} cm")

	p = figure(width = 1280, height = 720, x_range = (-0.5 , 30.5), y_range=(0, 2.04), x_axis_label="Distancia (cm)", y_axis_label="Transmitancia", tools=["pan", "wheel_zoom", "box_zoom", "reset", "save"])
	p.line(source=source, color=colors[-1], legend_label=f"n2: {n2:.5e} m²/W", line_width=2)

	if points != 0:
		p.title = f"Curva promedio a {laser_power} mW - Filtro SG cúbico con {points} puntos"
	else:
		p.title = f"Curva promedio a {laser_power} mW - Sin filtrado"

	p.title.align = "center"
	p.title.text_font_size = "20pt"

	p.xaxis.major_label_text_font_size = "20pt"
	p.xaxis.axis_label_text_font_size = "20pt"

	p.yaxis.major_label_text_font_size = "20pt"
	p.yaxis.axis_label_text_font_size = "20pt"

	p.legend.location = "top_right"
	p.legend.label_text_font_size = "20pt"

	p.y_range.start = min(minis) - 0.05
	p.y_range.end = max(maxis) + 0.05

	show(p)

	sleep(1)

	return source, n2

In [38]:
laser_wavelength = 532
sample_length = 1
points = 11
powers = [10, 20, 60]

colors = ["black", "blue", "green", "orange", "magenta", "red"]

In [39]:
pathsmdo = [
	["Final Measures/Medalla de Oro/[1] 10 mW.csv",
	 "Final Measures/Medalla de Oro/[2] 10 mW.csv",
	 "Final Measures/Medalla de Oro/[3] 10 mW.csv",
	 "Final Measures/Medalla de Oro/[4] 10 mW.csv",
	 "Final Measures/Medalla de Oro/[5] 10 mW.csv"],

	["Final Measures/Medalla de Oro/[1] 20 mW.csv",
	 "Final Measures/Medalla de Oro/[2] 20 mW.csv",
	 "Final Measures/Medalla de Oro/[3] 20 mW.csv",
	 "Final Measures/Medalla de Oro/[4] 20 mW.csv",
	 "Final Measures/Medalla de Oro/[5] 20 mW.csv"],

	["Final Measures/Medalla de Oro/[1] 60 mW.csv",
	 "Final Measures/Medalla de Oro/[2] 60 mW.csv",
	 "Final Measures/Medalla de Oro/[3] 60 mW.csv",
	 "Final Measures/Medalla de Oro/[4] 60 mW.csv",
	 "Final Measures/Medalla de Oro/[5] 60 mW.csv"]
]

sourcesmdo = []
n2mdo = []
for i in range(len(powers)):
		print(f"================ Laser power: {powers[i]} ================")
		tmp_source, tmp_n2 = doThePlot(pathsmdo[i], powers[i], points)
		sourcesmdo.append(tmp_source)
		n2mdo.append(tmp_n2)
		print("")

================ Laser power: 10 ================
n2 1: -3.2476345521837834e-11
n2 2: -3.411217278489293e-11
n2 3: -3.558084996342153e-11
n2 4: -3.176703940006781e-11
n2 5: -3.229154891623591e-11

n2: -3.229154891623591e-11 m²/W
ΔTpv: 0.555943208563282
Δzpv: -0.017799999999999993 cm



================ Laser power: 20 ================
n2 1: -3.435361426609448e-11
n2 2: -3.5686644984547675e-11
n2 3: -3.5344511978398687e-11
n2 4: -3.5308651845398124e-11
n2 5: -3.412867240517415e-11

n2: -3.412867240517415e-11 m²/W
ΔTpv: 1.1245997963265515
Δzpv: -0.018599999999999995 cm



================ Laser power: 60 ================
n2 1: -3.4933186195269483e-11
n2 2: -3.43624613610208e-11
n2 3: -3.678701868904608e-11
n2 4: -3.478315875108238e-11
n2 5: -3.357067815096364e-11

n2: -3.357067815096364e-11 m²/W
ΔTpv: 1.9350056439938952
Δzpv: -0.0319 cm


In [40]:
p = figure(width = 1280, height = 720, x_range = (-0.5 , 30.5), y_range=(0, 2.04), x_axis_label="Distancia (cm)", y_axis_label="Transmitancia", tools=["pan", "wheel_zoom", "box_zoom", "reset", "save"])

tricolor = ["red", "blue", "green"]
for i in range(len(powers)):
	p.line(source=sourcesmdo[i], color=tricolor[i], legend_label=f"{powers[i]} mW - n2: {n2mdo[i]:.5e} m²/W", line_width=2)

p.title = f"Curvas de transmitancia del aceite 1 (MdO)"

p.title.align = "center"
p.title.text_font_size = "20pt"

p.xaxis.major_label_text_font_size = "20pt"
p.xaxis.axis_label_text_font_size = "20pt"

p.yaxis.major_label_text_font_size = "20pt"
p.yaxis.axis_label_text_font_size = "20pt"

p.legend.location = "top_right"
p.legend.label_text_font_size = "20pt"

p.y_range.start = min(i for s in sourcesmdo for i in s["y"]) - 0.05
p.y_range.end = max(i for s in sourcesmdo for i in s["y"]) + 0.05

show(p)

In [ ]:
pathsexg = [
	["Final Measures/Gallo Clássico/[1] 10 mW.csv",
	 "Final Measures/Gallo Clássico/[2] 10 mW.csv",
	 "Final Measures/Gallo Clássico/[3] 10 mW.csv",
	 "Final Measures/Gallo Clássico/[4] 10 mW.csv",
	 "Final Measures/Gallo Clássico/[5] 10 mW.csv"],

	["Final Measures/Gallo Clássico/[1] 20 mW.csv",
	 "Final Measures/Gallo Clássico/[2] 20 mW.csv",
	 "Final Measures/Gallo Clássico/[3] 20 mW.csv",
	 "Final Measures/Gallo Clássico/[4] 20 mW.csv",
	 "Final Measures/Gallo Clássico/[5] 20 mW.csv"],

	["Final Measures/Gallo Clássico/[1] 60 mW.csv",
	 "Final Measures/Gallo Clássico/[2] 60 mW.csv",
	 "Final Measures/Gallo Clássico/[3] 60 mW.csv",
	 "Final Measures/Gallo Clássico/[4] 60 mW.csv",
	 "Final Measures/Gallo Clássico/[5] 60 mW.csv"]
]

sourcesexg = []
n2exg = []
for i in range(len(powers)):
		print(f"================ Laser power: {powers[i]} ================")
		tmp_source, tmp_n2 = doThePlot(pathsexg[i], powers[i], points)
		sourcesexg.append(tmp_source)
		n2exg.append(tmp_n2)
		print("")

================ Laser power: 10 ================
n2 1: -2.403985471807174e-11
n2 2: -2.270930099726445e-11
n2 3: -2.3883225422460283e-11
n2 4: -2.321188638037115e-11
n2 5: -2.3938030885854175e-11

n2: -2.3938030885854175e-11 m²/W
ΔTpv: 0.4642938065399467
Δzpv: -0.0158 cm



================ Laser power: 20 ================
n2 1: -2.8368506708735717e-11
n2 2: -2.051057479647613e-11
n2 3: -2.649176495341902e-11
n2 4: -2.219807905824548e-11
n2 5: -2.6578906198963244e-11

n2: -2.6578906198963244e-11 m²/W
ΔTpv: 0.8269179672204507
Δzpv: -0.0197 cm



================ Laser power: 60 ================
n2 1: -2.875711467618447e-11
n2 2: -2.6828020637827212e-11
n2 3: -3.05456871031534e-11
n2 4: -2.9646750106864414e-11
n2 5: -2.8065439855216927e-11

n2: -2.8065439855216927e-11 m²/W
ΔTpv: 1.7733386060148515
Δzpv: -0.029099999999999994 cm


In [42]:
p = figure(width = 1280, height = 720, x_range = (-0.5 , 30.5), y_range=(0, 2.04), x_axis_label="Distancia (cm)", y_axis_label="Transmitancia", tools=["pan", "wheel_zoom", "box_zoom", "reset", "save"])

tricolor = ["red", "blue", "green"]
for i in range(len(powers)):
	p.line(source=sourcesexg[i], color=tricolor[i], legend_label=f"{powers[i]} mW - n2: {n2exg[i]:.5e} m²/W", line_width=2)

p.title = f"Curvas de transmitancia del aceite 2 (EXG)"

p.title.align = "center"
p.title.text_font_size = "20pt"

p.xaxis.major_label_text_font_size = "20pt"
p.xaxis.axis_label_text_font_size = "20pt"

p.yaxis.major_label_text_font_size = "20pt"
p.yaxis.axis_label_text_font_size = "20pt"

p.legend.location = "top_right"
p.legend.label_text_font_size = "20pt"

p.y_range.start = min(i for s in sourcesexg for i in s["y"]) - 0.05
p.y_range.end = max(i for s in sourcesexg for i in s["y"]) + 0.05

show(p)

In [ ]:
pathsppex = [
	["Final Measures/Gallo PP3/[1] 10 mW.csv",
	 "Final Measures/Gallo PP3/[2] 10 mW.csv",
	 "Final Measures/Gallo PP3/[3] 10 mW.csv",
	 "Final Measures/Gallo PP3/[4] 10 mW.csv",
	 "Final Measures/Gallo PP3/[5] 10 mW.csv"],

	["Final Measures/Gallo PP3/[1] 20 mW.csv",
	 "Final Measures/Gallo PP3/[2] 20 mW.csv"],

	["Final Measures/Gallo PP3/[1] 60 mW.csv",
	 "Final Measures/Gallo PP3/[2] 60 mW.csv"]
]

sourcesppex = []
n2ppex = []
for i in range(len(powers)):
		print(f"================ Laser power: {powers[i]} ================")
		tmp_source, tmp_n2 = doThePlot(pathsppex[i], powers[i], points)
		sourcesppex.append(tmp_source)
		n2ppex.append(tmp_n2)
		print("")

================ Laser power: 10 ================
n2 1: -5.243926284916878e-10
n2 2: -5.761650352444166e-10
n2 3: -5.619902423938335e-10
n2 4: -4.867901726077813e-10
n2 5: -5.505251156154101e-10

n2: -5.505251156154101e-10 m²/W
ΔTpv: 2.452168181906294
Δzpv: -0.06880000000000001 cm



================ Laser power: 20 ================
n2 1: 4.266885263368863e-10
n2 2: 4.187397135165542e-10

n2: 4.187397135165542e-10 m²/W
ΔTpv: 1.1586758924530667
Δzpv: 0.2215 cm



================ Laser power: 60 ================
n2 1: 1.3123796420488976e-10
n2 2: 1.3298682682468135e-10

n2: 1.3298682682468135e-10 m²/W
ΔTpv: 1.100963369696866
Δzpv: 0.22209999999999996 cm
